In [6]:
import cv2
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import math
import shutil
import numpy as np
import re
import ntpath
import random

The main goal is to generate a dataset for classification of left right eye.

We use the source `classification_source_dataset.txt` which is the fusion of pngs.txt and pngs2.txt.

In [ ]:
def join_dataset(dataset1_file: str, dataset2_file: str, output_file: str):
    def strip(str_: str):
        return str_.strip()
    
    def is_exist(file):
        return Path(file).exists()
    
    if not Path(dataset1_file).exists() or not Path(dataset2_file).exists():
        return
        
    with open(dataset1_file, "r") as file1, \
         open(dataset2_file, "r") as file2, \
         open(output_file, "w") as output:
        
        lst1 = map(strip,file1.readlines())
        lst2 = map(strip,file2.readlines())
        
        # remove duplicate element
        s1 = set(lst1)
        s2 = set(lst2)
        mult_set = s1 | s2
                
        lst1 = sorted(mult_set)
        
        res = list(filter(is_exist, lst1))
        print(len(res))
        
        output.writelines(line + "\n" for line in res)
        
        
join_dataset("source/pngs.txt", "source/pngs2.txt", "source/classification_source_dataset.txt")

We have to split the dataset into 3 sub dataset (train, valid and test).
You can choose the repartition.

An heristic is:
- Subject more represented in the dataset should more be in the train and the test should contain a diversity of subject.

In [8]:
def _extract_subject_name(file: str):
    file = file.strip()
    
    # from a path get the basename
    filename = ntpath.basename(file)
    
    lst = filename.split('_')
    if len(lst) == 0:
        return
    
    subject_id = lst[1]
    
    # get the biggest substring of char
    sequences = re.findall(r'\D+', subject_id)
    return max(sequences, key=len, default="")

def get_histogram(files: list[str]) -> dict:
    """
        Do an histogram of name with the name in key and the number of eye with this name
    """
    hist = {}
    
    for file in files:
        name = _extract_subject_name(file)
        print(name)
        
        count = hist.get(name, 0)
        hist[name] = count + 1
    
    return hist

Algo's main part

In [12]:
def _fill_class(rng, hist: dict,
                values: np.ndarray, probabilities: np.ndarray, 
                quantity_goal: int, interval: int, timeout = 50):
    """
        use a timetout because greedy doesn't garanty a global convergence
    """
    result = []
    count_result = 0
    nb_values = values.size()
    
    invalid_draw = 0
    
    while invalid_draw < timeout:
        idx = rng.choice(values.size, p=probabilities)
        draw = values[idx]
        
        v = ((count_result + hist[draw]) / nb_values) - quantity_goal
        
        if v > quantity_goal + interval:
            invalid_draw += 1
            continue
        
        result.append(draw)
        count_result += hist[draw]

        np.delete(values, idx)
        np.delete(probabilities, idx)

        if v >= quantity_goal - interval:
            return result
            
    return result, count_result

def probabilistic_greedy_function(train, qtrain: int, 
                                  valid, qvalid: int,
                                  test, qtest: int,
                                  lambda_size = 1, lambda_diversity = 1):
    
    # E_size
    E_size = 0
    
    [_, n_image_train] = train
    [_, n_image_valid] = valid
    [vtest, n_image_test] = test
    
    n_image = n_image_train + n_image_test + n_image_valid

    E_size += abs((n_image_train / n_image) - qtrain)
    E_size += abs((n_image_valid / n_image) - qvalid)
    E_size += abs((n_image_test / n_image) - qtest)
    
    # E_diversity
    E_diversity = vtest.size() / n_image_test
    
    return lambda_size * E_size - lambda_diversity * E_diversity

def repeated_weighted_draw (hist: dict[str, int],
                           qtrain: float, qtest: float, 
                           itrain: float, itest: float,
                           draw_nb = 100):
    """
    :params qtrain, qvalid, qtest: 
        are the quantity of the dataset in each category and
    :params itrain, itest:
        are the degree of liberty around the quantity's goal
    """
        
    rng = np.random.default_rng()
    best_train = [None, -1]
    best_valid = [None, -1]
    best_test = [None, -1]
    best_quality = -1
    
    for _ in range(draw_nb):
        subjects = np.asarray(list(hist.keys()))
        count = np.asarray(list(hist.values()))
        
        train = np.asarray(subjects)
        ptrain = count / count.sum()

        # draw
        dtrain, train_count = _fill_class(rng, hist, train, ptrain, qtrain, itrain)
        
        test = np.asarray(train)
        ptest = 1 / ptrain
        
        dtest, test_count = _fill_class(rng, hist, test, ptest, qtest, itest)
        
        dvalid = test.copy()
        qvalid = 1 - qtest - qtrain
        valid_count = count.sum() - train_count - test_count
        
        # eval the quality of the split
        quality = probabilistic_greedy_function([dtrain, train_count], qtrain,
                                                [dvalid, valid_count], qvalid,
                                                [dtest, test_count], qtest)
        
        # maximise the quality
        if best_quality == -1 or best_quality < quality:
            best_train = [dtrain, train_count]
            best_valid = [dvalid, valid_count]
            best_test = [dtest, test_count]
            best_quality = quality
        
    return best_train, best_valid, best_test

In [13]:
def _log_subject_eye(train, valid, test):
    
    subject_train, nb_image_train = train
    subject_valid, nb_image_valid = valid
    subject_test, nb_image_test = test
    
    nb_subject_train = subject_train.size()
    nb_subject_valid = subject_valid.size()
    nb_subject_test = subject_test.size()
    
    total_subject = nb_subject_train + nb_subject_valid + nb_subject_test
    total_eyes = nb_image_train + nb_image_valid + nb_image_test

    s =  f"train : subject {nb_subject_train}, eye {nb_image_train}\n"
    s += f"valid : subject {nb_subject_valid}, eye {nb_image_valid}\n"
    s += f"test  : subject {nb_subject_test}, eye {nb_image_test}\n"
    s += f"total : subject {total_subject}, eye {total_eyes}\n"

    print(s)
    

def split_dataset(dataset: str, 
                  train = 0.75, valid = 0.15, test = 0.15, 
                  itrain = 0.05, itest = 0.02):
    
    if not Path(dataset).exists():
        return
    
    with open(dataset, "r") as db:
        files = db.readlines()
        
        hist = get_histogram(files)
        train, valid, test = repeated_weighted_draw(hist, train, test, itrain, itest)
        
        _log_subject_eye(train, valid, test)
        
        # split into 3 folder
        
    return

split_dataset("dataset/source/classification_source_dataset.txt")


DUJ
DUJ
DUJ
DUJ
DUJ
DUJ
DUJ
DUJ
DUJ
ATM
ATM
ATM
AUZ
AUZ
AUZ
BOM
BOM
BOM
CHC
CHC
CHC
CHC
CHC
CHC
GUJ
SUR
SUR
SUR
SUR
AIZ
AIZ
AIZ
AIZ
AIZ
AIZ
AUZ
AUZ
AUZ
AUZ
AUZ
AUZ
BOM
BOM
BOM
BOM
BOM
BOM
CAD
CAD
CAD
CAD
CAD
CAD
CHC
CHC
CHC
CHC
CHC
CHC
CHC
GUJ
GUJ
GUJ
GUJ
GUJ
GUJ
DUM
DUM
DUM
DUM
DUM
DUM
DUM
LEC
LEC
LEC
LEC
LEC
LEC
SAM
SAM
SAM
SAM
SAM
SAM
TAF
TAF
TAF
TAF
TAF
TAF
TAF
CHC
CHC
CHC
CHC
CHC
CHC
CHC
CHC
DAD
DAD
DAD
DAD
DAD
DAD
DAD
CAD
CAD
CAD
CAD
CAD
CAD
CHC
CHC
CHC
CHC
CHC
COS
COS
COS
COS
COS
COS
COS
COS
DAD
DAD
DAD
DAD
DAD
DAD
OLL
OLL
OLL
OLL
OLL
OLL
OLL
COS
FAM
FAM
FAM
FAM
FAM
FAM
DEC
DEC
DEC
DEC
AUZ
AUZ
AUZ
AUZ
AUZ
AUZ
BEM
BEM
BEM
BEM
BEM
BEM
BOM
BOM
BOM
BOM
BOM
BOM
BOM
FIY
FIY
FIY
FIY
FIY
FIY
GUJ
GUJ
GUJ
GUJ
GUJ
GUJ
GUJ
GUJ
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
TOM
BIF
BIF
BIF
BIF
BIF
BIF
DEC
DEC
DEC
DEC
DEC
DEC
JAM
JAM
JAM
JAM
POA
POA
POA
POA
THM
THM
THM
THM
THM
THM
AUZ
AUZ
AUZ
AUZ
AUZ
AUZ
COY
COY
COY
COY
COY
COY
COY
DAD
DAD
DAD
DAD
DAD
DAD
DEC
DEC
DEC
DEC
DEC
DEC


TypeError: 'int' object is not callable

In [3]:
import numpy as np

rng = np.random.default_rng()
valeurs = np.array(["rouge", "vert", "bleu"])
probas  = np.array([0.5, 0.3, 0.2])

tirage = rng.choice(valeurs, size=1, replace=True, p=probas)
print(tirage)   # ['rouge' 'bleu'] par exemple, jamais deux fois la même valeur
valeurs

['rouge']


array(['rouge', 'vert', 'bleu'], dtype='<U5')